# Matrix testing

In [ ]:
from sqlalchemy.orm import Session
from sqlalchemy import create_engine

DB_URL = "sqlite:///../../matchit.db"

engine = create_engine(DB_URL, echo=False)

session = Session(engine)

In [2]:
import sys

sys.path.append("../../")

In [3]:
from scipy.sparse import load_npz

similarity_ascents = load_npz("../../similarity_ascent.npz")
similarity_style = load_npz("../../similarity_style.npz")
similarity_grade = load_npz("../../similarity_grade.npz")

In [4]:
import numpy as np
from sqlalchemy import select

from models.boulder import Boulder


def recommend_boulders(
    input_boulders,
    top_n=5,
    ascent_weight=0.5,
    style_weight=0.25,
    grade_weight=0.25,
):

    ascents = similarity_ascents[:, input_boulders].sum(axis=1).A1
    style = similarity_style[:, input_boulders].sum(axis=1).A1
    grade = similarity_grade[:, input_boulders].sum(axis=1).A1

    # Remove input boulders from the recommendation
    ascents[input_boulders] = 0
    style[input_boulders] = 0
    grade[input_boulders] = 0

    sim_scores = (
        ascent_weight * ascents + style_weight * style + grade_weight * grade
    )

    best_boulders = np.argsort(-sim_scores)[:top_n]

    return best_boulders.tolist()


ids = recommend_boulders(
    [35240], top_n=20, ascent_weight=0.5, style_weight=0.25, grade_weight=0.25
)

boulders = session.execute(
    select(Boulder.name, Boulder.id).filter(Boulder.id.in_(ids))
).all()

boulder_dict = {
    boulder_id: boulder_name for boulder_name, boulder_id in boulders
}
boulder_names = [(boulder_dict[boulder_id], boulder_id) for boulder_id in ids]
display(boulder_names)

[('Le Pilier du Désert (assis)', 4902),
 ('Gecko (assis)', 16976),
 ('Khéops (assis)', 1940),
 ('The Island', 35237),
 ('Imothep (du sol)', 1753),
 ('Mammunk (assis)', 23901),
 ('Le Pied à Coulisse', 20185),
 ('La Ligne de Bête', 22848),
 ('Petite Sorcellerie', 5263),
 ('Le Marathon de Boissy', 22841),
 ('Gourmandise (direct) / The Traphouse', 1942),
 ('Quoi de Neuf', 41292),
 ('Bélial', 5),
 ('Mécanique Élémentaire', 15775),
 ('Mammunk', 23903),
 ('La Picharête', 16247),
 ('Délire Onirique (assis)', 15229),
 ('Jack Under the Box', 25018),
 ('Le Dernier Fléau', 1938),
 ("L'Arête de Boissy (assis)", 22847)]